# Analiza sieciowa scenariusza filmowego

## Wersja studencka: od promptu do działającego kodu

Ten notatnik jest szablonem do pracy w Google Colab na zajęciach z humanistyki cyfrowej. Nie zawiera gotowych rozwiązań. Zamiast tego prowadzi Cię przez serię małych promptów, które możesz przekazać modelowi AI, a następnie wkleić wygenerowany kod do pustej komórki.

Efektem końcowym będzie:
1. pobranie i oczyszczenie tekstu scenariusza,
2. wydzielenie scen i postaci,
3. zbudowanie tabeli współwystąpień,
4. narysowanie sieci relacji,
5. zapisanie plików do dalszej analizy.

Wersja z wcześniejszymi rozwiązaniami została zachowana w pliku `analiza_sieciowa_scenariusza_prowadzacy.ipynb`.


## Jak pracować z tym notatnikiem

1. Zmień adres scenariusza w komórce parametru.
2. Skopiuj prompt z bieżącego kroku do modelu AI.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj rezultat z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Jeśli wynik nie zgadza się z opisem, popraw tylko bieżący krok.
6. Dopiero po uzyskaniu poprawnego wyniku przejdź dalej.

W każdym kroku skupiamy się na jednym małym zadaniu. Nie próbuj rozwiązywać kilku etapów naraz. Najważniejszy wynik każdego kroku powinien pozostać dostępny do następnego kroku, ale to model ma zdecydować, jak to zorganizować w kodzie.


## Parametr startowy

To jedyna komórka, którą zmieniasz ręcznie przed rozpoczęciem pracy. Wybierz adres strony scenariusza z IMSDb.


In [11]:
# Jedyny parametr, który zmieniasz w tym notatniku
adres_scenariusza = "https://imsdb.com/scripts/Indiana-Jones-and-the-Raiders-of-the-Lost-Ark.html"


---
## Etap 1: Od strony internetowej do czystego tekstu

Najpierw upewniamy się, że pracujemy na właściwej stronie i że potrafimy oddzielić tekst scenariusza od pozostałych elementów witryny.


### Krok 1A. Pobranie strony i kontrola surowej odpowiedzi

#### Cel i sens analityczny

Zanim zaczniemy analizować scenariusz, trzeba potwierdzić, że wskazany adres rzeczywiście prowadzi do strony z treścią, na której da się pracować dalej.


#### Prompt dla modelu

```text
Kontekst:
Chcesz sprawdzić, czy adres podany w komórce parametru rzeczywiście prowadzi do strony ze scenariuszem filmowym.

Wejście:
Adres scenariusza zapisany w pierwszej komórce parametru.

Zadanie:
Pobierz zawartość tej strony i przygotuj krótki podgląd surowej odpowiedzi. Zachowaj pobraną treść tak, aby można było wykorzystać ją w następnym kroku bez ponownego pobierania.

Pokaż wynik:
- komunikat, czy pobranie się udało,
- końcowy adres strony po ewentualnym przekierowaniu,
- pierwsze 8-10 niepustych linii pobranej treści.

Warunek poprawności:
W wyniku powinny być widoczne elementy strony internetowej, a treść nie może być pusta.

Jeśli wystąpi błąd:
Wyświetl krótką informację, czy problem dotyczy adresu, braku połączenia albo pustej odpowiedzi.

Nie rób jeszcze:
Nie wydobywaj tekstu scenariusza i nie analizuj jego struktury.
```


In [12]:
import requests

# Jedyny parametr, który zmieniasz w tym notatniku
adres_scenariusza = "https://imsdb.com/scripts/Indiana-Jones-and-the-Raiders-of-the-Lost-Ark.html"

def sprawdz_strone(url):
    try:
        # Pobieramy stronę (dodajemy nagłówek User-Agent, aby uniknąć zablokowania)
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=15)

        # Sprawdzamy przekierowania i status
        if response.status_code == 200:
            print("✅ Sukces: Strona została pobrana pomyślnie.")
            print(f"🔗 Końcowy adres strony: {response.url}\n")

            # Zapisanie do zmiennej globalnej dla następnych kroków w notatniku
            global surowa_tresc
            surowa_tresc = response.text

            if not surowa_tresc.strip():
                print("❌ Błąd: Odpowiedź serwera jest pusta.")
                return

            print("Podgląd surowej odpowiedzi (pierwsze 10 niepustych linii):")
            print("-" * 50)

            # Ekstrakcja pierwszych 8-10 niepustych linii (widać tu tagi HTML)
            niepuste_linie = [linia.strip() for linia in surowa_tresc.splitlines() if linia.strip()]
            for linia in niepuste_linie[:10]:
                print(linia)

            print("-" * 50)
            print("Zmienna 'surowa_tresc' jest gotowa do użycia w kolejnym kroku.")

        else:
            print(f"❌ Błąd: Nie udało się pobrać strony (Kod HTTP: {response.status_code}).")

    except requests.exceptions.MissingSchema:
        print("❌ Błąd: Niepoprawny format adresu URL. Upewnij się, że zawiera http:// lub https://")
    except requests.exceptions.ConnectionError:
        print("❌ Błąd: Brak połączenia z serwerem. Sprawdź internet lub poprawność adresu.")
    except requests.exceptions.Timeout:
        print("❌ Błąd: Przekroczono czas oczekiwania na odpowiedź serwera.")
    except Exception as e:
        print(f"❌ Błąd: Wystąpił nieoczekiwany problem: {e}")

# Wywołanie funkcji
sprawdz_strone(adres_scenariusza)

✅ Sukces: Strona została pobrana pomyślnie.
🔗 Końcowy adres strony: https://imsdb.com/scripts/Indiana-Jones-and-the-Raiders-of-the-Lost-Ark.html

Podgląd surowej odpowiedzi (pierwsze 10 niepustych linii):
--------------------------------------------------
<html>
<head>
<!-- Google tag (gtag.js) -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-W5BXG8HCH3"></script>
<script>
window.dataLayer = window.dataLayer || [];
function gtag(){dataLayer.push(arguments);}
gtag('js', new Date());
gtag('config', 'G-W5BXG8HCH3');
</script>
--------------------------------------------------
Zmienna 'surowa_tresc' jest gotowa do użycia w kolejnym kroku.


#### Po uruchomieniu powinieneś zobaczyć


- Krótki komunikat potwierdzający, że strona została pobrana albo czytelne wyjaśnienie problemu.
- Końcowy adres strony, jeśli po drodze nastąpiło przekierowanie.
- Fragment surowej treści, który wygląda jak pełna strona internetowa, a nie jak pusty wynik.


### Krok 1B. Wydobycie właściwego tekstu scenariusza

#### Cel i sens analityczny

Strona zawiera menu, linki i elementy pomocnicze. Do dalszej analizy potrzebny jest tylko właściwy tekst scenariusza.


#### Prompt dla modelu

```text
Kontekst:
Surowa strona zawiera dużo elementów, które nie należą do scenariusza.

Wejście:
Pobrana treść strony z poprzedniego kroku.

Zadanie:
Wydobądź tylko właściwy tekst scenariusza i odrzuć menu, nagłówki, stopki, linki i inne elementy witryny. Zachowaj oczyszczony tekst tak, aby był dostępny w następnym kroku.

Pokaż wynik:
- krótki komunikat, że tekst scenariusza został odnaleziony,
- pierwsze 15 niepustych linii oczyszczonego tekstu.

Warunek poprawności:
W podglądzie powinien być widoczny układ scenariusza lub dialogów, a nie nawigacja strony.

Jeśli wystąpi błąd:
Wyjaśnij, że nie udało się odnaleźć głównego bloku z tekstem scenariusza albo że wynik nadal zawiera zbyt dużo treści pobocznych.

Nie rób jeszcze:
Nie licz znaków i nie rozpoznawaj scen ani postaci.
```


In [14]:
!pip install beautifulsoup4

from bs4 import BeautifulSoup

def wydobadz_scenariusz(html):
    if not html:
        print("❌ Błąd: Brak surowej treści. Upewnij się, że poprzednia komórka została wykonana poprawnie.")
        return

    try:
        soup = BeautifulSoup(html, 'html.parser')

        # W serwisie IMSDb tekst scenariusza zazwyczaj znajduje się w komórce z klasą 'scrtext'
        blok_scenariusza = soup.find('td', class_='scrtext')

        # Plan awaryjny: jeśli nie ma klasy 'scrtext', szukamy największego bloku <pre>
        if not blok_scenariusza:
            bloki_pre = soup.find_all('pre')
            if bloki_pre:
                blok_scenariusza = max(bloki_pre, key=lambda b: len(b.get_text()))

        if blok_scenariusza:
            # Pobieramy sam tekst, ignorując tagi HTML, ale zachowując oryginalne formatowanie
            tekst = blok_scenariusza.get_text()

            # Weryfikacja, czy odnaleziony tekst jest wystarczająco długi (by nie złapać stopki)
            if len(tekst.strip()) < 500:
                print("❌ Błąd: Odnaleziony blok tekstu jest zbyt krótki. Wynik nadal może zawierać zbyt dużo treści pobocznych.")
                return

            print("✅ Sukces: Tekst scenariusza został odnaleziony i oczyszczony z elementów strony.\n")

            # Zapisanie do zmiennej globalnej dla następnych kroków
            global oczyszczony_tekst
            oczyszczony_tekst = tekst

            print("Podgląd oczyszczonego tekstu (pierwsze 15 niepustych linii):")
            print("-" * 50)

            # Ekstrakcja pierwszych 15 niepustych linii z zachowaniem wcięć
            niepuste_linie = [linia for linia in oczyszczony_tekst.splitlines() if linia.strip()]
            for linia in niepuste_linie[:15]:
                print(linia.rstrip())

            print("-" * 50)
            print("Zmienna 'oczyszczony_tekst' jest gotowa do użycia w kolejnym kroku.")

        else:
            print("❌ Błąd: Nie udało się odnaleźć głównego bloku z tekstem scenariusza.")

    except Exception as e:
        print(f"❌ Błąd podczas parsowania HTML: {e}")

# Sprawdzamy, czy zmienna z poprzedniego kroku istnieje i uruchamiamy funkcję
if 'surowa_tresc' in globals():
    wydobadz_scenariusz(surowa_tresc)
else:
    print("❌ Błąd: Zmienna 'surowa_tresc' nie istnieje. Uruchom najpierw komórkę z poprzedniego kroku.")



✅ Sukces: Tekst scenariusza został odnaleziony i oczyszczony z elementów strony.

Podgląd oczyszczonego tekstu (pierwsze 15 niepustych linii):
--------------------------------------------------
                                "RAIDERS OF THE LOST ARK"
                                      Screenplay by
                                     Lawrence Kasdan
                                         Story by
                                       George Lucas
               FADE IN:
               EXT. PERU - HIGH JUNGLE - DAY
               The dense, lush rain forests of the eastern slopes of the
               Andes, the place known as "The Eyebrow of the Jungle". Ragged,
               jutting canyon walls are half-hidden by the thick mists.
               The MAIN TITLE is followed by this:
                                           PERU
                                           1936
               A narrow trail across the green face of the canyon. A group
               of men make 

#### Po uruchomieniu powinieneś zobaczyć


- Potwierdzenie, że udało się odseparować właściwy tekst scenariusza.
- Kilkanaście pierwszych linii przypominających zapis scenariusza filmowego.
- Brak widocznego menu, stopki i innych elementów witryny.


### Krok 1C. Podgląd tekstu i szybka kontrola jakości

#### Cel i sens analityczny

Zanim przejdziemy dalej, warto upewnić się, że tekst scenariusza jest wystarczająco długi i wygląda wiarygodnie.


#### Prompt dla modelu

```text
Kontekst:
Przed analizą scen i postaci trzeba sprawdzić, czy oczyszczony tekst wygląda kompletnie.

Wejście:
Oczyszczony tekst scenariusza z poprzedniego kroku.

Zadanie:
Pokaż krótki podgląd tekstu oraz podstawowe informacje o jego rozmiarze. Jeśli tekst wygląda podejrzanie krótko albo zawiera głównie elementy witryny, wypisz ostrzeżenie. Zachowaj tekst do dalszej analizy.

Pokaż wynik:
- liczbę znaków,
- liczbę linii,
- fragment początku tekstu o długości mniej więcej 1000-1500 znaków.

Warunek poprawności:
Podgląd powinien przypominać początek scenariusza, a nie przypadkowy fragment strony internetowej.

Jeśli wystąpi błąd:
Wyświetl informację, że tekst jest pusty, zbyt krótki albo nadal wymaga oczyszczenia.

Nie rób jeszcze:
Nie rozpoznawaj scen i postaci.
```


In [16]:
def sprawdz_kompletnosc(tekst):
    if not tekst:
        print("❌ Błąd: Brak tekstu do analizy. Upewnij się, że poprzedni krok zakończył się sukcesem.")
        return

    # Obliczanie podstawowych statystyk
    liczba_znakow = len(tekst)
    linie = tekst.splitlines()
    liczba_linii = len(linie)

    print("📊 Podstawowe informacje o tekście:")
    print(f" - Liczba znaków: {liczba_znakow:,}".replace(',', ' '))
    print(f" - Liczba linii: {liczba_linii:,}".replace(',', ' '))
    print()

    # Weryfikacja kompletności
    if liczba_znakow < 20000 or liczba_linii < 1000:
        print("⚠️ OSTRZEŻENIE: Tekst wydaje się podejrzanie krótki jak na pełny scenariusz filmu.\n")
    else:
        print("✅ Sukces: Objętość tekstu wygląda na poprawną i wskazuje na kompletny scenariusz.\n")

    # Podgląd fragmentu tekstu (~1200 znaków)
    limit_znakow = 1200
    if len(tekst) > limit_znakow:
        fragment = tekst[:limit_znakow]
        ostatni_enter = fragment.rfind('\n')
        if ostatni_enter != -1:
            fragment = fragment[:ostatni_enter]
        fragment += "\n\n[... dalsza część tekstu ukryta w podglądzie ...]"
    else:
        fragment = tekst

    print(f"Podgląd początku scenariusza (~{limit_znakow} znaków):")
    print("-" * 50)
    print(fragment)
    print("-" * 50)
    print("Zmienna 'oczyszczony_tekst' jest gotowa do kolejnego kroku.")

# Uruchomienie sprawdzania
if 'oczyszczony_tekst' in globals():
    sprawdz_kompletnosc(oczyszczony_tekst)
else:
    print("❌ Błąd: Zmienna 'oczyszczony_tekst' nie istnieje. Uruchom najpierw krok 2.")

📊 Podstawowe informacje o tekście:
 - Liczba znaków: 248 076
 - Liczba linii: 5 260

✅ Sukces: Objętość tekstu wygląda na poprawną i wskazuje na kompletny scenariusz.

Podgląd początku scenariusza (~1200 znaków):
--------------------------------------------------

                                "RAIDERS OF THE LOST ARK"

                                      Screenplay by

                                     Lawrence Kasdan

                                         Story by

                                       George Lucas

                

               FADE IN:

               EXT. PERU - HIGH JUNGLE - DAY

               The dense, lush rain forests of the eastern slopes of the 
               Andes, the place known as "The Eyebrow of the Jungle". Ragged, 
               jutting canyon walls are half-hidden by the thick mists.

               The MAIN TITLE is followed by this:

                                           PERU
                                           1936

 

#### Po uruchomieniu powinieneś zobaczyć


- Dwie podstawowe liczby opisujące wielkość materiału: znaki i linie.
- Dłuższy fragment początku scenariusza, który da się ocenić wzrokowo.
- W razie problemu: ostrzeżenie, że materiał nie nadaje się jeszcze do dalszych kroków.


---
## Etap 2: Rozpoznanie struktury scenariusza

W tym etapie przechodzimy od ciągłego tekstu do uporządkowanej struktury: najpierw sceny, potem postacie, a na końcu pełna lista scen z przypisanymi bohaterami.


### Krok 2A. Wykrycie granic scen

#### Cel i sens analityczny

Scena jest podstawową jednostką analizy sieciowej. Najpierw musimy odnaleźć miejsca, w których scenariusz przechodzi do nowej sceny.


#### Prompt dla modelu

```text
Kontekst:
W scenariuszach filmowych nowe sceny są zwykle otwierane krótkimi nagłówkami opisującymi miejsce i czas akcji, często zaczynającymi się od skrótów oznaczających wnętrze lub plener.

Wejście:
Oczyszczony tekst scenariusza.

Zadanie:
Wykryj granice scen i zbuduj uporządkowaną listę scen z numerami oraz nagłówkami. Zachowaj ten wynik do kolejnego kroku.

Pokaż wynik:
- łączną liczbę rozpoznanych scen,
- pierwszych 8 rekordów w formie `numer sceny | nagłówek sceny`.

Warunek poprawności:
Nagłówki powinny wyglądać jak opisy miejsca i czasu akcji, a nie jak dialog lub didaskalia. Jeśli scenariusz ma niestandardowy zapis, krótko opisz przyjętą zasadę rozpoznawania scen.

Jeśli wystąpi błąd:
Wyjaśnij, że nie udało się znaleźć wyraźnych granic scen albo że wykryte nagłówki są zbyt niejednoznaczne.

Nie rób jeszcze:
Nie przypisuj postaci do scen.
```


In [17]:
import re

def wykryj_sceny(tekst):
    if not tekst:
        print("❌ Błąd: Brak tekstu scenariusza do analizy. Upewnij się, że poprzedni krok wykonał się poprawnie.")
        return

    # Wyrażenie regularne dopasowujące standardowe nagłówki scen w języku angielskim
    # Szukamy linii zaczynających się od INT., EXT., INT/EXT., EXT/INT., I/E.
    wzorzec_sceny = r'^\s*(?:\d+[\.\s]+)?\b(INT\.|EXT\.|INT/EXT\.|EXT/INT\.|INT\./EXT\.|EXT\./INT\.|I/E\.)\s+.*$'

    sceny = []

    for linia in tekst.splitlines():
        linia_clean = linia.strip()
        # Nagłówki scen w standardowym formacie są w całości pisane wielkimi literami
        if re.match(wzorzec_sceny, linia_clean, re.IGNORECASE):
            sceny.append(linia_clean)

    if not sceny:
        print("❌ Błąd: Nie udało się odnaleźć wyraźnych granic scen.")
        print("Wyjaśnienie: Brak linii zaczynających się od standardowych skrótów (INT., EXT.). Wynik jest zbyt niejednoznaczny.")
        return

    # Zbudowanie struktury z numerami scen dla kolejnych kroków
    global lista_scen
    lista_scen = [
        {"numer": idx + 1, "naglowek": naglowek}
        for idx, naglowek in enumerate(sceny)
    ]

    print("✅ Sukces: Wykryto granice scen i zbudowano uporządkowaną listę.\n")
    print(f"📊 Łączna liczba rozpoznanych scen: {len(lista_scen)}")
    print("\nPrzyjęta zasada rozpoznawania scen:")
    print("Jako nagłówek rozpoznawana jest każda linia zaczynająca się od skrótów oznaczenia miejsca akcji (INT., EXT., INT/EXT., EXT/INT.).\n")

    print("Podgląd pierwszych 8 rekordów (numer sceny | nagłówek sceny):")
    print("-" * 70)
    for scena in lista_scen[:8]:
        print(f"{scena['numer']:>3} | {scena['naglowek']}")
    print("-" * 70)
    print("\nZmienna 'lista_scen' została zachowana i jest gotowa do wykorzystania w kolejnym kroku.")

# Uruchomienie funkcji z tekstem z poprzedniego kroku
if 'oczyszczony_tekst' in globals():
    wykryj_sceny(oczyszczony_tekst)
else:
    print("❌ Błąd: Zmienna 'oczyszczony_tekst' nie istnieje. Uruchom najpierw poprzedni krok.")

✅ Sukces: Wykryto granice scen i zbudowano uporządkowaną listę.

📊 Łączna liczba rozpoznanych scen: 139

Przyjęta zasada rozpoznawania scen:
Jako nagłówek rozpoznawana jest każda linia zaczynająca się od skrótów oznaczenia miejsca akcji (INT., EXT., INT/EXT., EXT/INT.).

Podgląd pierwszych 8 rekordów (numer sceny | nagłówek sceny):
----------------------------------------------------------------------
  1 | EXT. PERU - HIGH JUNGLE - DAY
  2 | INT. TEMPLE - INCLINED PASSAGE - DAY
  3 | EXT. FRONT OF THE TEMPLE - DAY
  4 | EXT. THE JUNGLE - INDY'S RUN - VARIOUS SHOTS - DAY
  5 | EXT. THE URUBAMBA RIVER - DUSK
  6 | INT. JOCK'S PLANE - DUSK
  7 | EXT. JOCK'S PLANE - TWILIGHT
  8 | INT. INDY'S OFFICE, SMALL EASTERN COLLEGE - DAY
----------------------------------------------------------------------

Zmienna 'lista_scen' została zachowana i jest gotowa do wykorzystania w kolejnym kroku.


#### Po uruchomieniu powinieneś zobaczyć


- Liczbę scen rozpoznanych w całym tekście.
- Krótką listę pierwszych nagłówków, które wyglądają jak kolejne miejsca lub momenty akcji.
- Ewentualną krótką notatkę, jeśli scenariusz używa nietypowego formatu nagłówków.


### Krok 2B. Wykrycie kandydatów na nazwy postaci

#### Cel i sens analityczny

Sieć relacji budujemy z udziałem postaci, więc trzeba najpierw odróżnić ich nazwy od innych elementów zapisu scenariusza.


#### Prompt dla modelu

```text
Kontekst:
Nazwy postaci w scenariuszach często są zapisane wielkimi literami, wyraźnie odsunięte od lewego marginesu i czasem mają dopiski w nawiasach, które trzeba usunąć.

Wejście:
Oczyszczony tekst scenariusza.

Zadanie:
Znajdź kandydatów na nazwy postaci, oczyść je z dopisków w nawiasach i odfiltruj oczywiste instrukcje reżyserskie, przejścia montażowe oraz inne elementy, które nie są postaciami. Zachowaj wynik do kolejnego kroku.

Pokaż wynik:
- liczbę unikalnych kandydatów po oczyszczeniu,
- próbkę 20 rekordów w formie `oryginalny zapis | oczyszczona nazwa`.

Warunek poprawności:
Na liście powinny dominować imiona lub nazwy postaci, a nie komendy typu przejście, opis kamery albo tytuł karty.

Jeśli wystąpi błąd:
Pokaż kilka problematycznych przykładów i krótko wyjaśnij, dlaczego trudno je rozstrzygnąć.

Nie rób jeszcze:
Nie przypisuj postaci do konkretnych scen.
```


In [18]:
# tu wklej kod wygenerowany przez model
import re

def wydobadz_postaci(tekst):
    if not tekst:
        print("❌ Błąd: Brak tekstu scenariusza do analizy.")
        return

    # Wyrażenie regularne usuwające tekst w nawiasach (wraz z przylegającymi spacjami)
    wzorzec_nawiasow = r'\s*\(.*?\)\s*'

    # Słowa i frazy typowe dla instrukcji reżyserskich i przejść montażowych, które trzeba odrzucić
    wykluczenia = [
        'CUT TO', 'FADE IN', 'FADE OUT', 'DISSOLVE', 'CONTINUOUS',
        'TITLE', 'ANGLE', 'CLOSE UP', 'WIDE SHOT', 'POV', 'REVERSE',
        'THE END', 'SCENE', 'OMITTED', 'PAN', 'ZOOM', 'TRACKING',
        'INT.', 'EXT.', 'TO:', 'MATCH CUT', 'INSERT', 'BACK TO', 'DAY', 'NIGHT'
    ]

    znalezione_postaci = {}

    for linia in tekst.splitlines():
        # Zamieniamy ewentualne tabulatory na spacje, by poprawnie policzyć wielkość wcięcia
        linia_spacje = linia.expandtabs(4)
        linia_bez_spacji = linia_spacje.strip()

        if not linia_bez_spacji:
            continue

        wciecie = len(linia_spacje) - len(linia_spacje.lstrip())

        # Warunki dla postaci:
        # 1. Duże wcięcie (zwykle między 15 a 45 spacji)
        # 2. Krótka linia (wyklucza wyśrodkowane, długie akapity tekstowe)
        # 3. Same WIELKIE LITERY
        if 15 <= wciecie <= 45 and len(linia_bez_spacji) < 40 and linia_bez_spacji.isupper():

            # Odrzucamy linie kończące się dwukropkiem (np. CUT TO:)
            if linia_bez_spacji.endswith(':'):
                continue

            # Sprawdzamy, czy linia zawiera słowa z listy wykluczeń
            czy_wykluczyc = any(wykluczenie in linia_bez_spacji for wykluczenie in wykluczenia)

            if not czy_wykluczyc:
                # Oczyszczamy z nawiasów (np. "INDY (V.O.)" -> "INDY")
                oczyszczona_nazwa = re.sub(wzorzec_nawiasow, '', linia_bez_spacji).strip()

                # Dodatkowa weryfikacja: unikamy pojedynczych liter i samych cyfr
                if oczyszczona_nazwa and len(oczyszczona_nazwa) > 1 and not re.fullmatch(r'\d+', oczyszczona_nazwa):

                    # Zapisujemy w słowniku dla unikalności.
                    # Jeśli postać "INDY" już istnieje, nie dodajemy jej ponownie.
                    if oczyszczona_nazwa not in znalezione_postaci:
                        znalezione_postaci[oczyszczona_nazwa] = linia_bez_spacji

    if not znalezione_postaci:
        print("❌ Błąd: Nie odnaleziono kandydatów na postacie.")
        print("Przykład problemu: Ten scenariusz może używać niestandardowych wcięć lub nie zapisywać bohaterów WIELKIMI LITERAMI, co uniemożliwia proste ich oddzielenie od opisów akcji.")
        return

    # Przygotowanie zmiennej do zachowania w pamięci
    global unikalne_postaci
    unikalne_postaci = [
        {"oczyszczona": k, "oryginalna": v}
        for k, v in znalezione_postaci.items()
    ]

    # Sortowanie alfabetyczne według oczyszczonej nazwy dla lepszej czytelności
    unikalne_postaci.sort(key=lambda x: x['oczyszczona'])

    print("✅ Sukces: Odnaleziono i oczyszczono kandydatów na postacie.\n")
    print(f"📊 Liczba unikalnych kandydatów (po oczyszczeniu): {len(unikalne_postaci)}\n")

    print("Podgląd próbki 20 rekordów (oryginalny zapis | oczyszczona nazwa):")
    print("-" * 65)
    for postac in unikalne_postaci[:20]:
        print(f"{postac['oryginalna']:<35} | {postac['oczyszczona']}")
    print("-" * 65)
    print("\nZmienna 'unikalne_postaci' jest zachowana i gotowa do użycia w kolejnym kroku.")

# Uruchomienie funkcji z oczyszczonym tekstem z poprzednich kroków
if 'oczyszczony_tekst' in globals():
    wydobadz_postaci(oczyszczony_tekst)
else:
    print("❌ Błąd: Zmienna 'oczyszczony_tekst' nie istnieje. Uruchom najpierw poprzednie kroki.")

✅ Sukces: Odnaleziono i oczyszczono kandydatów na postacie.

📊 Liczba unikalnych kandydatów (po oczyszczeniu): 55

Podgląd próbki 20 rekordów (oryginalny zapis | oczyszczona nazwa):
-----------------------------------------------------------------
"DANGER - MUNITIONS"                | "DANGER - MUNITIONS"
"RAIDERS OF THE LOST ARK"           | "RAIDERS OF THE LOST ARK"
AMIR                                | AMIR
ARAB BARTENDER                      | ARAB BARTENDER
ARMY INTEL. #9906753                | ARMY INTEL. #9906753
AT THE CHAMBER OF LIGHT             | AT THE CHAMBER OF LIGHT
AT THE EDGE OF THE CLEARING         | AT THE EDGE OF THE CLEARING
AT THE VINED LANDING                | AT THE VINED LANDING
BARRANCA                            | BARRANCA
BELLOQ                              | BELLOQ
BELZIG                              | BELZIG
BRODY                               | BRODY
CHAMBER OF LIGHT                    | CHAMBER OF LIGHT
CLIMBER                             | CLIMBER
DO NO

#### Po uruchomieniu powinieneś zobaczyć


- Liczbę rozpoznanych nazw po oczyszczeniu.
- Próbkę zapisów pokazującą, jak surowe nazwy zostały uproszczone do postaci możliwych do porównywania.
- Brak dominacji elementów technicznych scenariusza nad rzeczywistymi postaciami.


### Krok 2C. Zbudowanie listy scen z postaciami

#### Cel i sens analityczny

Dopiero po połączeniu scen i postaci otrzymujemy strukturę, z której można obliczać współwystąpienia.


#### Prompt dla modelu

```text
Kontekst:
Masz już rozpoznane granice scen oraz kandydatów na nazwy postaci.

Wejście:
Wynik kroku z granicami scen i wynik kroku z rozpoznanymi postaciami.

Zadanie:
Zbuduj finalną listę scen. Każda scena ma zawierać numer, nagłówek oraz zbiór unikalnych postaci obecnych w tej scenie. Jeśli w danej scenie nie uda się rozpoznać żadnej postaci, zaznacz to jasno. Zachowaj listę scen do dalszych obliczeń.

Pokaż wynik:
- łączną liczbę scen,
- 5 przykładowych rekordów w formie `numer sceny | nagłówek | postacie`.

Warunek poprawności:
Ta sama postać nie powinna być powtórzona kilka razy w obrębie jednej sceny.

Jeśli wystąpi błąd:
Wyświetl krótką informację, czy problem dotyczy scen bez postaci, czy niespójności między wcześniejszymi krokami.

Nie rób jeszcze:
Nie buduj jeszcze rankingu postaci ani relacji między nimi.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Liczbę scen gotowych do dalszej analizy.
- Pięć czytelnych rekordów pokazujących numer sceny, jej nagłówek i listę rozpoznanych postaci.
- Jasną informację, jeśli część scen pozostała bez przypisanych postaci.


---
## Etap 3: Które postacie są najważniejsze?

Zanim zbudujemy sieć relacji, sprawdzamy rozkład obecności postaci w scenach. To prosty, ale bardzo użyteczny punkt wyjścia do interpretacji.


### Krok 3A. Ranking postaci według liczby scen

#### Cel i sens analityczny

Liczba scen, w których pojawia się dana postać, daje pierwszą miarę jej znaczenia w strukturze narracji.


#### Prompt dla modelu

```text
Kontekst:
Masz listę scen, z których każda zawiera unikalne postacie.

Wejście:
Finalna lista scen z przypisanymi postaciami.

Zadanie:
Policz, w ilu różnych scenach pojawia się każda postać. Posortuj wynik od najwyższego do najniższego. Zachowaj ranking do następnego kroku.

Pokaż wynik:
- łączną liczbę unikalnych postaci,
- tabelę 20 najwyższych wyników w formie `postać | liczba scen`.

Warunek poprawności:
W obrębie jednej sceny dana postać może zostać policzona tylko raz.

Jeśli wystąpi błąd:
Wyświetl informację, że lista scen jest pusta albo że nie udało się rozpoznać postaci w wystarczającej liczbie scen.

Nie rób jeszcze:
Nie rysuj wykresu i nie buduj relacji między postaciami.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Krótką tabelę rankingową z najczęściej występującymi postaciami.
- Łączną liczbę rozpoznanych postaci.
- Wyniki posortowane malejąco według liczby scen.


#### Pytanie interpretacyjne

Które postacie dominują w strukturze scen i czy ten ranking zgadza się z Twoją intuicją o bohaterach filmu?


### Krok 3B. Wykres 10 najczęściej występujących postaci

#### Cel i sens analityczny

Wizualizacja pozwala szybko ocenić, czy scenariusz jest skupiony wokół jednej postaci, czy raczej rozkłada uwagę na kilka figur.


#### Prompt dla modelu

```text
Kontekst:
Masz już gotowy ranking liczby scen przypisanych do postaci.

Wejście:
Ranking postaci z poprzedniego kroku.

Zadanie:
Narysuj czytelny poziomy wykres dla 10 postaci o największej liczbie scen. Dodaj tytuł oraz opisy osi. Zachowaj ranking do dalszej pracy.

Pokaż wynik:
- pojedynczy wykres obejmujący tylko 10 najwyższych wyników,
- czytelne nazwy postaci,
- porządek od najwyższego do najniższego wyniku.

Warunek poprawności:
Jeśli ranking zawiera mniej niż 10 postaci, narysuj tyle, ile jest dostępnych, i wyraźnie to zaznacz.

Jeśli wystąpi błąd:
Wyświetl krótkie wyjaśnienie, dlaczego nie da się przygotować wykresu.

Nie rób jeszcze:
Nie buduj jeszcze relacji współwystąpień.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Jeden poziomy wykres z nazwami postaci i długościami słupków odpowiadającymi liczbie scen.
- Wyraźny tytuł i podpisane osie.
- Brak dodatkowych elementów, które utrudniałyby szybkie odczytanie rankingu.


#### Pytanie interpretacyjne

Czy rozkład obecności postaci wygląda na skoncentrowany wokół jednej osi fabularnej, czy bardziej rozproszony między kilka ważnych postaci?


---
## Etap 4: Od scen do relacji między postaciami

Tutaj przechodzimy od obecności postaci do ich współwystępowania. To właśnie z tych danych powstanie sieć relacji.


### Krok 4A. Pary współwystąpień wewnątrz pojedynczych scen

#### Cel i sens analityczny

Każda scena z co najmniej dwiema postaciami dostarcza zestawu potencjalnych relacji, które można później zagregować w całym scenariuszu.


#### Prompt dla modelu

```text
Kontekst:
Relacja między dwiema postaciami pojawia się wtedy, gdy obie występują w tej samej scenie.

Wejście:
Lista scen z unikalnymi postaciami.

Zadanie:
Dla każdej sceny wygeneruj wszystkie unikalne pary postaci występujących razem. Zachowaj wynik tak, aby można było później zsumować go dla całego scenariusza.

Pokaż wynik:
- liczbę scen, z których udało się utworzyć przynajmniej jedną parę,
- próbkę 15 wierszy w formie `numer sceny | postać A | postać B`.

Warunek poprawności:
Para nie może łączyć postaci z samą sobą, a ta sama para nie powinna pojawiać się podwójnie w obrębie jednej sceny. Zachowaj stały porządek zapisu par, aby później łatwo je sumować.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem wynika z tego, że sceny mają zbyt mało postaci, czy z niespójności wcześniejszych danych.

Nie rób jeszcze:
Nie sumuj jeszcze wyników dla całego scenariusza.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Informację, z ilu scen udało się utworzyć relacje.
- Próbkę pojedynczych par współwystąpień przypisanych do numerów scen.
- Brak duplikatów tej samej pary w obrębie jednej sceny.


### Krok 4B. Tabela relacji dla całego scenariusza

#### Cel i sens analityczny

Po zsumowaniu wszystkich par otrzymujemy podstawową tabelę krawędzi, z której można już budować sieć.


#### Prompt dla modelu

```text
Kontekst:
Masz listę par współwystąpień wygenerowanych dla poszczególnych scen.

Wejście:
Wynik kroku z parami współwystąpień.

Zadanie:
Zsumuj wszystkie powtarzające się pary w całym scenariuszu i utwórz tabelę relacji. Każda relacja ma opisywać, jak często dana para postaci występuje razem. Zachowaj tę tabelę do etapów sieciowych.

Pokaż wynik:
- liczbę unikalnych relacji,
- 15 najsilniejszych wierszy w formie `postać A | postać B | siła relacji`.

Warunek poprawności:
Ta sama relacja nie może pojawiać się kilka razy pod różnymi wariantami kolejności zapisu.

Jeśli wystąpi błąd:
Wyświetl krótką informację, że nie udało się utworzyć relacji albo że wynik jest pusty.

Nie rób jeszcze:
Nie buduj jeszcze sieci i nie zapisuj plików.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Liczbę unikalnych relacji między postaciami.
- Tabelę pokazującą najsilniejsze duety wraz z siłą ich współwystępowania.
- Wyniki posortowane od najsilniejszej do najsłabszej relacji.


#### Pytanie interpretacyjne

Które duety wydają się centralne dla narracji i czy ich siła wynika raczej z długotrwałej osi fabuły, czy z wielu powracających spotkań?


---
## Etap 5: Sieć, podgląd i eksport danych

Na końcu zamieniamy tabelę relacji w sieć, oglądamy jej prosty podgląd i zapisujemy pliki do dalszej analizy poza Colabem.


### Krok 5A. Zbudowanie sieci relacji

#### Cel i sens analityczny

To moment przejścia od tabeli relacji do właściwej reprezentacji sieciowej, na której można liczyć miary i rysować strukturę powiązań.


#### Prompt dla modelu

```text
Kontekst:
Tabela relacji opisuje już, które postacie są połączone i jak silne jest to połączenie.

Wejście:
Tabela relacji z poprzedniego kroku.

Zadanie:
Zbuduj sieć, w której węzłami są postacie, a połączenia opisują ich współwystępowanie. Siła relacji ma zostać zachowana przy każdym połączeniu. Zachowaj gotową sieć do następnego kroku.

Pokaż wynik:
- liczbę węzłów,
- liczbę połączeń,
- krótką listę 10 postaci z największą liczbą połączeń.

Warunek poprawności:
Każda relacja z tabeli powinna pojawić się w sieci dokładnie raz.

Jeśli wystąpi błąd:
Wyjaśnij, że tabela relacji jest pusta albo niespójna i dlatego nie da się zbudować sieci.

Nie rób jeszcze:
Nie rysuj jeszcze grafu i nie zapisuj plików.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Dwie podstawowe liczby opisujące wielkość sieci: węzły i połączenia.
- Krótką listę najlepiej połączonych postaci.
- Brak wizualizacji na tym etapie: chodzi tylko o potwierdzenie, że sieć została zbudowana poprawnie.


### Krok 5B. Podgląd grafu w notatniku

#### Cel i sens analityczny

Wizualny podgląd pomaga zobaczyć, czy sieć ma jedno centrum, kilka grup albo bardziej rozproszoną strukturę.


#### Prompt dla modelu

```text
Kontekst:
Masz już zbudowaną sieć relacji między postaciami.

Wejście:
Gotowa sieć z poprzedniego kroku.

Zadanie:
Narysuj prosty podgląd sieci bezpośrednio w notatniku. Niech bardziej połączone postacie będą lepiej widoczne, a silniejsze relacje wyraźniejsze. Dodaj etykiety przynajmniej dla najważniejszych postaci.

Pokaż wynik:
- pojedynczy rysunek sieci z tytułem,
- czytelne etykiety najważniejszych postaci,
- różnicę widoczną między słabszymi i silniejszymi relacjami.

Warunek poprawności:
Jeśli sieć jest bardzo gęsta, ogranicz liczbę etykiet albo uprość podgląd i krótko wyjaśnij ten wybór, zamiast tworzyć nieczytelny obraz.

Jeśli wystąpi błąd:
Wyświetl informację, że sieć jest zbyt mała albo zbyt gęsta, aby dało się ją pokazać w czytelnej formie bez dodatkowego uproszczenia.

Nie rób jeszcze:
Nie zapisuj jeszcze plików wyjściowych.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Jeden rysunek sieci umieszczony bezpośrednio pod komórką.
- Większą widoczność postaci silniej osadzonych w sieci oraz wyraźniejsze relacje o większej sile.
- Krótką notę wyjaśniającą uproszczenie, jeśli sieć okaże się bardzo gęsta.


#### Pytanie interpretacyjne

Czy na podglądzie widać jednego głównego pośrednika, kilka grup postaci czy raczej gęstą strukturę bez wyraźnego centrum?


### Krok 5C. Eksport danych do dalszej analizy

#### Cel i sens analityczny

Ostatni krok przygotowuje dwa pliki, które można pobrać z Colab i analizować dalej w innych narzędziach.


#### Prompt dla modelu

```text
Kontekst:
Masz gotową tabelę relacji oraz zbudowaną sieć postaci.

Wejście:
Tabela relacji i gotowa sieć.

Zadanie:
Zapisz tabelę relacji do pliku `krawedzie.csv`, a całą sieć do pliku `graf_postaci.graphml`. Przed zapisaniem pliku GraphML upewnij się, że każdy węzeł ma ustawiony atrybut label, który zawiera czytelną nazwę postaci. Jeśli nazwa postaci jest już użyta jako identyfikator węzła, ustaw label na wartość identyczną z ID. Na końcu pokaż krótko, jakie pliki powstały i gdzie mozna je znaleźć w środowisku notatnika.

Pokaż wynik:
- komunikat o zapisaniu obu plików,
- listę nazw zapisanych plików,
- krótką wskazówkę, jak pobrać je z Colab.

Warunek poprawności:
Oba pliki powinny być niepuste i gotowe do otwarcia poza notatnikiem.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się zapisać i z jakiego powodu.

Nie rób jeszcze:
Nie wykonuj już dalszej analizy.
```


In [ ]:
# tu wklej kod wygenerowany przez model


#### Po uruchomieniu powinieneś zobaczyć


- Potwierdzenie zapisania plików `krawedzie.csv` i `graf_postaci.graphml`.
- Krótką listę lub komunikat pokazujący, że pliki są widoczne w środowisku notatnika.
- Jednozdaniową wskazówkę, jak pobrać pliki z Google Colab.


---
## Co dalej?

- Otwórz `graf_postaci.graphml` w Gephi albo innym narzędziu do analizy sieci, aby policzyć miary centralności i sprawdzić społeczności postaci.
- Otwórz `krawedzie.csv` w arkuszu kalkulacyjnym, jeśli chcesz filtrować, sortować i porównywać relacje bez pracy na grafie.
- Zmień adres scenariusza w komórce parametru i porównaj strukturę narracyjną różnych filmów.
- Jeśli któryś krok działa źle, poproś model o poprawienie tylko tego jednego fragmentu zamiast generowania całego rozwiązania od nowa.
